In [10]:
# Walk up to the repo root (the folder with run.py), chdir there, add it to sys.path

import os, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / 'run.py').exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('run.py not found; open this notebook inside the repo')
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print('repo root:', ROOT)

repo root: C:\Users\Admin\Desktop\player-intelligence-system-cpe342


In [11]:
# Load Task 3 and check the main facts

import numpy as np, pandas as pd
from pis.common import DEFAULT_DATA # ../cpe342-kaggle-main
tr = pd.read_csv(DEFAULT_DATA / '03_regression' / 'train.csv') # train
y = tr['spending_30d']
print('zero share', round((y == 0).mean(), 3), '| skew', round(y.skew(), 2),
      '| log1p skew', round(np.log1p(y).skew(), 2))
big = y > 20000
print('> 20k:', round(big.mean(), 3), 'of rows,', round(y[big].sum() / y.sum(), 3), 'of spend')
print(tr.groupby('segment')['spending_30d'].apply(lambda s: round((s == 0).mean(), 3)))

zero share 0.481 | skew 4.17 | log1p skew 0.31
> 20k: 0.124 of rows, 0.914 of spend
segment
0.0    0.724
1.0    0.480
2.0    0.306
3.0    0.071
Name: spending_30d, dtype: float64


In [12]:
# Where does the main model make its error? (no training needed)

o = pd.read_csv('baseline/task3_oof.csv')
o['err'] = (o['target'] - o['prediction']).abs()
bins, names = [-1, 0, 1000, 20000, 1e9], ['0', '1-1k', '1k-20k', '>20k']
o['bucket'] = pd.cut(o['target'], bins, labels=names)
summary = o.groupby('bucket', observed=True)['err'].agg(['count', 'sum'])
summary['share_of_error'] = (summary['sum'] / summary['sum'].sum()).round(3)
summary

,count,sum,share_of_error
bucket,,,
0,50049,9.117431e+07,0.288
1-1k,16620,4.896170e+06,0.015
1k-20k,24478,2.050960e+07,0.065
>20k,12853,1.997970e+08,0.632
